# 02. ARC Baseline v1: Evaluation & Failure Analysis

**ARC Prize 2026 Research Project**  
This notebook analyzes the performance of our first deterministic baseline solver (`RuleBasedSearchSolver_v1`) on the held-out ARC evaluation benchmark.

### Architecture of Baseline v1:
```
ARC Task Demonstration Pairs
       ↓
Candidate Generator (Dihedral geometric, Deduced color substitutions, Bounding crops, Fractals, Gravity, Symmetry)
       ↓
Exact-Match Verifier (Filters candidates that perfectly explain 100% of training pairs)
       ↓
Candidate Ranker (Occam's razor / simplicity scoring)
       ↓
Test Prediction Generator (Applies top candidate to unseen test input)
```

### Objectives:
1. Measure task-level and test-pair exact-match accuracy without leaking test labels
2. Quantify the generalization gap (training demonstration fit vs. test generalization)
3. Analyze and categorize failure modes
4. Visualize solved tasks and investigate near-miss / spurious correlation tasks

In [ ]:
%matplotlib inline
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Add project root to Python path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_dataset, load_task
from src.data.visualizer import plot_task
from src.evaluation.harness import evaluate_dataset
from src.solvers.rule_based import RuleBasedSearchSolver

print("Modules imported successfully!")

## 1. Load Baseline Evaluation Results
We load the structured JSON benchmark results generated from `experiments/baseline_v1.py`.

In [ ]:
results_file = project_root / "results" / "baseline_v1.json"

if results_file.exists():
    with open(results_file, "r", encoding="utf-8") as f:
        results = json.load(f)
    print(f"Loaded results from: {results_file}")
else:
    print("Results file not found. Running baseline evaluation on ARC-AGI-1 evaluation set...")
    eval_dataset = load_dataset(project_root / "data" / "ARC-AGI-1" / "data" / "evaluation")
    solver = RuleBasedSearchSolver()
    results = evaluate_dataset(solver, eval_dataset)

print(f"\nSolver: {results['solver_name']}")
print(f"Total Tasks Evaluated: {results['total_tasks_evaluated']}")
print(f"Tasks Solved: {results['tasks_solved']} ({results['task_level_accuracy_pct']:.2f}%)")
print(f"Test-Pair Accuracy: {results['test_pair_accuracy_pct']:.2f}%")
print(f"Training Consistency: {results['train_consistency_pct']:.2f}%")
print(f"Generalization Gap: {results['generalization_gap_pct']:.2f}%")
print(f"Avg Latency per Task: {results['avg_runtime_ms_per_task']:.2f} ms")

## 2. Accuracy & Generalization Metrics Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy Comparison Bar Chart
metrics = ["Train Consistency", "Test-Pair Accuracy", "Task-Level Accuracy"]
values = [
    results["train_consistency_pct"],
    results["test_pair_accuracy_pct"],
    results["task_level_accuracy_pct"],
]
colors = ["#0074D9", "#2ECC40", "#FF851B"]

bars = ax1.bar(metrics, values, color=colors, edgecolor="black", width=0.5)
ax1.set_title("Baseline v1: Accuracy & Generalization Gap", fontweight="bold")
ax1.set_ylabel("Percentage (%)")
ax1.set_ylim(0, max(values) + 10)
for bar, val in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{val:.2f}%", ha="center", fontweight="bold")

# Failure Breakdown Pie Chart
breakdown = results["failure_breakdown"]
labels = list(breakdown.keys())
counts = list(breakdown.values())
pie_colors = ["#2ECC40" if k == "SOLVED" else ("#FF4136" if k == "GENERALIZATION_ERROR" else "#AAAAAA") for k in labels]

ax2.pie(
    counts,
    labels=labels,
    autopct="%1.1f%%",
    startangle=140,
    colors=["#2ECC40", "#FF4136", "#FFDC00", "#0074D9", "#AAAAAA"][:len(labels)],
)
ax2.set_title("Task Outcome Distribution", fontweight="bold")

plt.tight_layout()
plt.show()

## 3. Visualizing Solved ARC Tasks
Here we inspect tasks where the solver successfully discovered the true underlying transformation rule.

In [ ]:
solved_ids = results.get("solved_task_ids", [])
solved_rules = results.get("solved_task_rules", {})

print(f"Total Solved Tasks: {len(solved_ids)}")
display_count = min(5, len(solved_ids))

for i in range(display_count):
    tid = solved_ids[i]
    rule = solved_rules.get(tid, "N/A")
    print(f"\n=== Solved Task {i+1}/{display_count}: {tid} ===")
    print(f"Discovered Rule: {rule}")
    task_files = list((project_root / "data").rglob(f"{tid}.json"))
    if task_files:
        task = load_task(task_files[0])
        fig = plot_task(task)
        plt.show()

## 4. Spurious Correlation & Generalization Errors
Tasks where a candidate rule fit all training examples, but failed on the test example (overfitting on demonstration examples).

In [ ]:
gen_error_tasks = [
    r for r in results.get("per_task_results", [])
    if r.get("outcome") == "GENERALIZATION_ERROR"
]

print(f"Found {len(gen_error_tasks)} tasks with generalization errors.\n")

for r in gen_error_tasks[:3]:
    tid = r["task_id"]
    rule = r.get("chosen_rule")
    print(f"Task: {tid} | Candidate Fit on Train: {rule}")
    task_files = list((project_root / "data").rglob(f"{tid}.json"))
    if task_files:
        task = load_task(task_files[0])
        fig = plot_task(task)
        plt.show()

## 5. Summary of Baseline v1 Insights & Roadmap to v2

| Finding | Impact on ARC Solving | Research Direction for Baseline v2 |
| :--- | :--- | :--- |
| **Fixed Transformation Limits** | Pure global transformations only cover ~3-5% of ARC tasks | Need composable Domain Specific Language (DSL) with local operators |
| **Generalization Errors** | With only 2-3 demonstrations, simple rules can spuriously fit train pairs | Need invariant validation and minimum description length (MDL) priors |
| **Object Topology Absence** | Tasks requiring interior flood fill or gravity can't be represented as static matrix ops | Need object-centric perception and connected component segmentation |
| **Fast Execution** | Solver runs in <1 ms per task | Massive headroom for larger program search budgets and LLM proposal guidance |